# A.4: Round-trip tests and common bugs

Round-trip unusual strings, add special tokens twice, and truncate a long input.

Earlier imports and definitions this listing relies on are included below so the notebook runs on its own.


In [ ]:
# Imports and definitions from earlier listings, included so this notebook runs alone.

# From A.1 tiktoken basics
import tiktoken

# From A.2 Loading and training Hugging Face tokenizers
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("gpt2")

enc_out = tok("Hello world")

print(enc_out["input_ids"], tok.decode(enc_out["input_ids"]), tok.is_fast)


In [ ]:
tests = ["🙂🚀 and café", "中文 日本語 한국어", "x\u0301\u0302", "\t\t  indented\n\n\n",
         "3.14159265358979323846", "a\x00b"]
for name in ["gpt2", "o200k_base"]:
    e = tiktoken.get_encoding(name)
    print(name, all(e.decode(e.encode(t)) == t for t in tests))
bert = AutoTokenizer.from_pretrained("bert-base-uncased")
for t in tests[:3]:
    print(repr(t), "->", repr(bert.decode(bert(t, add_special_tokens=False)["input_ids"])))


**Expected output**

Output:

```text
gpt2 True
o200k_base True
'🙂🚀 and café' -> '[UNK] and cafe'
'中文 日本語 한국어' -> '中 文 日 本 語 한국어'
'x́̂' -> 'x'
```


In [ ]:
print(bert.convert_ids_to_tokens(bert("[CLS] hello world [SEP]")["input_ids"]))
print(bert.convert_ids_to_tokens(bert("hello world")["input_ids"]))


**Expected output**

Output:

```text
['[CLS]', '[CLS]', 'hello', 'world', '[SEP]', '[SEP]']
['[CLS]', 'hello', 'world', '[SEP]']
```


In [ ]:
print(tok.model_max_length, len(tok("word " * 2000, truncation=True)["input_ids"]))


**Expected output**

Output:

```text
1024 1024
```
